In [1]:
# ================================================================
#  LCCN-R24a - SAME-SPLIT TRANSLITERATION BASELINES: ARMS, ACCOUNTING,
#  CLASSICAL AND DEEP MACRO-F1  [CPU, ~1.5 h]
#  Answers IJIES Reviewer 2, point 4 (recent same-split baselines).
#  Three standalone transliteration arms on the raw tokens, each
#  rewriting ONLY tokens tagged romanized_kannada, nothing else:
#    indicxlit           AI4Bharat IndicXlit (neural, 2023) via the frozen
#                        cache xlit_cache.csv (covers 26,399 of 26,406 raw
#                        romanized types) - the control for lccn_indicxlit
#    itrans              indic_transliteration.sanscript ITRANS (rule-based)
#                        - the control for lccn_itrans
#    aksharamukha_itrans Aksharamukha, ITRANS scheme to Kannada (rule-based,
#                        maintained 2024-2025)
#  The pipeline's own build_arms() constructs indicxlit and itrans, so the
#  arm texts are the certified ones. Accounting per arm (training split:
#  forms rewritten, merged pairs = distinct roman forms sharing one output,
#  X(f), precision; full corpus: tokens changed). Then LR, SVM, NB, RF, XGB
#  with trainer v1.4; BiLSTM and CNN-BiLSTM run in R24b on GPU.
#  Writes xlit_texts.csv (id + text_* for all 40,000 comments) for R24b.
#  Add Input: paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml (output,
#             trainer.py), final-emotion-dataset, gold-ds,
#             transliteration-output (dataset: xlit_cache.csv).
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library indic-transliteration aksharamukha xgboost')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library indic-transliteration aksharamukha xgboost
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 992.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v3.4   /kaggle/input/datasets/tejaswinina/transliteration-output/pipeline.py
  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - THE THREE ARMS (certified build_arms for indicxlit / itrans)
# ================================================================
from collections import defaultdict, Counter
from functools import lru_cache
xlit = load_xlit(df, TXT)                      # needs xlit_cache.csv attached
assert isinstance(xlit, dict) and "indicxlit" in xlit, "xlit_cache.csv not found: attach transliteration-output"
A, _, _ = build_arms(norm_map, F2E, evidence, xlit, ["indicxlit", "itrans"], LEX=LEX)

from aksharamukha import transliterate as ak
@lru_cache(maxsize=None)
def _ak(w):
    try:
        out = ak.process("ITRANS", "Kannada", w)
        return out if is_kn(out) else w
    except Exception: return w
def aksharamukha_itrans(text):
    toks, tags = process_comment(text)
    return " ".join(_ak(w.lower()) if k == "romanized_kannada" else w for w, k in zip(toks, tags))
A["aksharamukha_itrans"] = aksharamukha_itrans

ARMS_NEW = ["indicxlit", "itrans", "aksharamukha_itrans"]
single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
train_mask = df[ID].isin(SPLIT["train"]).values
rows = []
for name in ARMS_NEW:
    df[f"text_{name}"] = df[TXT].map(A[name])
    ch = tot = 0
    for a, b in zip(df["text_raw"], df[f"text_{name}"]):
        ra, rb = a.lower().split(), b.lower().split(); assert len(ra) == len(rb), name
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    mp = defaultdict(Counter)
    for a, b in zip(df.loc[train_mask, "text_raw"], df.loc[train_mask, f"text_{name}"]):
        for w, x in zip(a.split(), b.split()): mp[w.lower()][x.lower()] += 1
    out = {f: c.most_common(1)[0][0] for f, c in mp.items()}
    rewritten = sum(1 for f, o in out.items() if o != f)
    kn_types = sum(1 for f, o in out.items() if o != f and is_kn(o))
    buckets = defaultdict(set)
    for f, o in out.items(): buckets[o].add(f)
    pairs = lab = same = 0
    for forms in buckets.values():
        n = len(forms)
        if n < 2: continue
        pairs += n*(n-1)//2
        ce = Counter(single[f] for f in forms if f in single); L = sum(ce.values())
        lab += L*(L-1)//2; same += sum(k*(k-1)//2 for k in ce.values())
    x = lab - same; prec = 1 - x/lab if lab else float("nan")
    rows.append(dict(arm=name, tokens_changed_full_pct=round(100*ch/tot, 2), forms_rewritten_train=rewritten,
                     forms_to_kannada_script=kn_types, merged_pairs_train=pairs, labelled_pairs_train=lab,
                     x_cross_train=x, merge_precision_train=round(prec, 4)))
    print(f"{name:<22} tok {100*ch/tot:5.2f}%  forms_rew {rewritten:>6,} (to Kannada script {kn_types:,})  "
          f"pairs {pairs:>7,}  X {x:>5,}  prec {prec:.4f}")
acc = pd.DataFrame(rows); acc.to_csv("/kaggle/working/xlit_accounting.csv", index=False)
df[[ID] + [f"text_{n}" for n in ARMS_NEW]].to_csv("/kaggle/working/xlit_texts.csv", index=False, encoding="utf-8-sig")
print("\nwritten xlit_texts.csv and xlit_accounting.csv")
print("Reference from the submitted paper: lccn_itrans and lccn_indicxlit token change 46-47% class; indicxlit alone should be close to lccn_indicxlit minus the LCCN rewrites.")


transliteration backend: ITRANS (indic_transliteration.sanscript)
converted 26,391 of 26,406 romanized types to Kannada script (99.9%)
IndicXlit cache: 26,742 entries (99.9% to Kannada script)
cache covers 26,399 of 26,406 romanized types (100.0%)

transliteration backends ready: ['itrans', 'indicxlit']
indicxlit              tok 21.19%  forms_rew 20,313 (to Kannada script 20,313)  pairs   5,244  X    21  prec 0.9717
itrans                 tok 21.19%  forms_rew 20,319 (to Kannada script 20,319)  pairs   1,462  X     3  prec 0.9885
aksharamukha_itrans    tok 21.19%  forms_rew 20,319 (to Kannada script 20,319)  pairs   1,546  X     3  prec 0.9887

written xlit_texts.csv and xlit_accounting.csv
Reference from the submitted paper: lccn_itrans and lccn_indicxlit token change 46-47% class; indicxlit alone should be close to lccn_indicxlit minus the LCCN rewrites.


In [5]:
# ================================================================
#  CELL 4 - CLASSICAL MACRO-F1 (trainer v1.4) WITH ANCHORS
# ================================================================
import shutil, glob, os, importlib
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)); assert tc
shutil.copy(tc[0], "/kaggle/working/trainer.py"); sys.modules.pop("trainer", None)
import trainer; importlib.reload(trainer); from trainer import *
assert trainer.TRAINER_VERSION == "1.4"; trainer.fingerprint()
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
NEW = [f"text_{n}" for n in ARMS_NEW]
lut = {c: dict(zip(df[ID].astype(str), df[c])) for c in NEW}
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for c in NEW: f[c] = ids.map(lut[c]); assert f[c].notna().all(), c
ARMS = ["raw", "lccn", "lccn_itrans", "lccn_indicxlit"] + ARMS_NEW
res = run_grid(tr, va, te, ARMS, ["Logistic Regression", "SVM", "Naive Bayes", "Random Forest", "XGBoost"],
               tag="lccn_r24_xlit_classical", resume=False)
print("\n=== classical macro-F1 ===")
print(res.to_string(index=False))
print("\nanchors raw / lccn / lccn_itrans / lccn_indicxlit must equal Table 3 (classical rows).")
print("DONE - xlit_texts.csv, xlit_accounting.csv, results_lccn_r24_xlit_classical.csv written; attach this output to R24b.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

=== Logistic Regression / raw ===
  -> macro-F1 0.8033

=== Logistic Regression / lccn ===
  -> macro-F1 0.8017

=== Logistic Regres